In [ ]:
import sys
sys.path.insert(0, '..')

from src.pipeline import SRRagPipeline

pipeline = SRRagPipeline(
    persist_dir='../vector_store',
    collection_name='sr_papers',
    llm_provider='mock',
)

info = pipeline.info()
print('Pipeline info:')
for k, v in info.items():
    print(f'  {k}: {v}')

In [ ]:
def show(result):
    print('=' * 65)
    print(f"Q: {result['question']}")
    print('=' * 65)
    print(result['answer_full'])
    print()
    print(f"Provider    : {result['provider']} / {result['model']}")
    print(f"Retrieval   : {result['retrieval_ms']:.0f} ms")
    print(f"Generation  : {result['generation_ms']:.0f} ms")
    print(f"Total       : {result['total_ms']:.0f} ms")
    tokens = result['token_usage']
    print(f"Tokens      : {tokens['total_tokens']} (est. cost ${tokens['estimated_cost_usd']:.6f})")
    print(f"Cites valid : {result['citations_valid']}")
    print()

In [ ]:
result = pipeline.query("What loss function does SRGAN use?")
show(result)

In [ ]:
result = pipeline.query("How does SwinIR use transformer attention?")
show(result)

In [ ]:
result = pipeline.query("What is the difference between SRGAN and ESRGAN?")
show(result)

In [ ]:
result = pipeline.query_with_filter(
    "What is the architecture?",
    method="RCAN",
    top_k=5,
)
show(result)

In [ ]:
result = pipeline.query_with_filter(
    "How is attention used for image restoration?",
    year_from=2021,
    top_k=5,
)
show(result)

In [ ]:
result = pipeline.query(
    "Which papers evaluate on DIV2K?",
    top_k=4,
    include_raw_chunks=True,
)
print(f"Retrieved {len(result['raw_chunks'])} chunks:\n")
for chunk in result['raw_chunks']:
    print(f"  [{chunk['citation_index']}] {chunk['method']} | p{chunk['page_number']} | score={chunk['score']:.3f}")

In [ ]:
import pandas as pd

questions = [
    "What loss function does SRGAN use?",
    "How does EDSR differ from SRResNet?",
    "What datasets does RCAN evaluate on?",
    "Explain the RRDB architecture in ESRGAN.",
    "What makes RealESRGAN handle real-world images?",
    "How does SwinIR use window-based attention?",
    "What is residual dense network in RDN?",
    "Which papers use adversarial training?",
]

rows = []
for q in questions:
    r = pipeline.query(q, top_k=5)
    rows.append({
        'Question':      q[:50],
        'Sources':       len(r['sources']),
        'Tokens':        r['token_usage']['total_tokens'],
        'Total ms':      r['total_ms'],
        'Cites valid':   r['citations_valid'],
    })

pd.DataFrame(rows)

In [ ]:
print("Pipeline end-to-end test complete.")
print(f"Total chunks in collection : {pipeline.info()['total_chunks']}")
print(f"LLM provider               : {pipeline.info()['llm_provider']}")
print()
print("Next step: Day 9 — Streamlit UI wraps this pipeline.")
print("Switch provider to 'openai' or 'gemini' in .env when ready.")